# B300 1개에서 추론 소프트웨어와 성능 측정

Inference Engineering 4장 실습이다. 프레임워크로 직접 모델을 실행하는 방법, 추론 엔진을 사용하는 방법, HTTP 서버에 부하를 주는 방법을 구분한다.

한 Linux 서버 또는 컨테이너에 B300 GPU 1개가 보이는 구성을 전제로 한다. 직접 실행과 서버 실험 모두 GPU 1개와 텐서 병렬화(TP) 1을 사용한다. B300에서 동작하는 PyTorch·CUDA·vLLM이 설치되어 있어야 한다.

이 노트북과 두 Python 파일이 있는 폴더를 작업 디렉터리로 사용한다. 기존 vLLM 환경의 Python 커널을 선택한다. 전체 셀을 한 번에 실행하지 않고, 서버 실행 안내에서 멈춰 별도 터미널에 서버를 시작한 뒤 계속한다.

측정값은 셀 출력에 표시한다. 노트북을 저장하면 출력도 함께 저장된다. 결과만 읽을 때는 셀을 실행하지 않는다. 별도 결과 파일은 작성하지 않는다.

## 1. 실행 설정

두 환경에서 같은 모델 가중치와 토크나이저를 사용한다. 기본 모델은 `Qwen/Qwen3.8-27B`, 정밀도는 BF16이다. 모델 이름은 두 환경에 준비한 동일한 로컬 모델 경로로 바꿀 수 있다. 모델이 없으면 Hugging Face에서 내려받는다.

27B 언어 모델 가중치는 BF16 기준 약 54GB이며 캐시와 실행 작업 공간에는 추가 메모리가 필요하다. 직접 실행 비교에 사용하는 GPU 1개의 가용 메모리를 확인한다. Qwen3.8-27B와 아래 CLI 옵션을 지원하는 환경이 필요하며, 패키지 준비 조건은 README를 참고한다.

`SINGLE_GPU`와 `SERVER_GPUS`에는 할당받은 GPU 번호를 지정한다. 아래 기본값은 노트북 프로세스에서 해당 GPU 번호를 사용할 수 있다고 가정한다. 서버 실험은 기존 작업이 사용하지 않는 GPU에서 실행한다.

서버 측정은 조건마다 요청 256개, 워밍업 요청 8개, 본 측정 3회를 사용한다. 서버와 벤치마크 클라이언트는 같은 머신에서 실행한다.

In [ ]:
import json
import os
import shlex
import subprocess
import sys
from urllib.request import Request, urlopen

MODEL = "Qwen/Qwen3.8-27B"
SINGLE_GPU = "0"
SERVER_GPUS = "0"
TENSOR_PARALLEL_SIZE = 1
BASE_URL = "http://127.0.0.1:8000"
NUM_PROMPTS = 256
REPEATS = 3

VLLM = [sys.executable, "-m", "vllm.entrypoints.cli.main"]

## 2. 프레임워크와 추론 엔진 직접 실행

PyTorch는 GPU 텐서 연산을 실행하고, Transformers는 모델 구조와 생성 API를 제공한다. vLLM은 추론 실행과 요청 스케줄링을 제공한다. 두 Python 파일을 열어 모델 로딩과 생성 호출을 비교한다.

Qwen3.8-27B의 공식 설정에 따라 Transformers 로더는 `Qwen3_5ForConditionalGeneration`을 사용한다. Transformers는 시각 인코더도 로드하지만 텍스트만 입력하고, vLLM은 `language_model_only=True`로 시각 인코더를 제외한다. 이 실험에서는 두 경로의 GPU 메모리 사용량을 비교하지 않는다. vLLM의 동시 처리 상한은 배치 크기에 맞춰 8개로 지정한다.

두 구현 모두 GPU 1개에서 같은 512토큰 입력 8개를 한 번에 처리하고, 각 요청에서 128토큰을 생성한다. 따라서 측정마다 실제 출력 토큰 합계가 1,024여야 한다. 입력은 토큰 길이를 통제하기 위한 반복 문장이며 답변 품질 평가용 데이터가 아니다.

모델 로딩과 입력 토큰화는 측정에서 제외한다. 같은 작업을 2회 준비 실행한 다음 3회 측정한다. Transformers는 최소·최대 생성 길이를 모두 128로 지정하고, vLLM은 `ignore_eos=True`를 사용해 출력 길이를 고정한다. vLLM의 접두사 캐시와 문자열 변환을 끄고 토큰 생성 API 호출 시간을 비교한다. PyTorch에서는 GPU 작업 완료를 기다리는 `torch.cuda.synchronize()`를 측정 전후에 호출한다.

두 프로그램은 각각 별도 프로세스로 실행되어, 프로그램이 끝나면 사용한 GPU 메모리가 해제된다. 아래 셀은 모델을 실제로 실행한다.

In [ ]:
single_gpu_env = os.environ.copy()
single_gpu_env["CUDA_VISIBLE_DEVICES"] = SINGLE_GPU

result = subprocess.run(
    [sys.executable, "transformers_baseline.py", MODEL],
    env=single_gpu_env,
    stdout=subprocess.PIPE,
    text=True,
    check=True,
)
print("Transformers: elapsed_s, output_tokens, output_tokens_per_s")
print(result.stdout.split("elapsed_s, output_tokens, output_tokens_per_s\n", 1)[1])

In [ ]:
result = subprocess.run(
    [sys.executable, "vllm_baseline.py", MODEL],
    env=single_gpu_env,
    stdout=subprocess.PIPE,
    text=True,
    check=True,
)
print("vLLM: elapsed_s, output_tokens, output_tokens_per_s")
print(result.stdout.split("elapsed_s, output_tokens, output_tokens_per_s\n", 1)[1])

### 직접 실행 결과 해석

세 번의 처리 시간과 출력 토큰 처리량을 비교한다. 이 값은 배치 8개를 처리한 전체 결과이며 개별 요청의 TTFT가 아니다. 직접 실행 결과에는 HTTP 통신과 서버 대기시간이 포함되지 않는다.

모델 실행 구현, 커널, 스케줄링 등 여러 차이가 포함된다. Qwen3.8-27B는 선형 어텐션과 전체 어텐션을 함께 사용하므로 차이를 PagedAttention 하나의 효과로 설명하지 않는다. 이 모델과 단일 배치 결과를 다른 모델 크기나 서비스 부하에 그대로 적용하지 않는다.

## 3. vLLM 서버 실행

아래 셀은 서버 명령을 출력한다. 출력된 명령을 같은 머신의 별도 터미널에 복사해 실행한다. 서버 시작이 완료된 뒤 다음 셀로 이동한다. 명령은 노트북 커널과 같은 Python 실행 파일을 사용한다.

서버는 `127.0.0.1:8000`에서만 요청을 받는다. TP 1은 B300 GPU 1개에서 모델을 실행한다.

반복 입력의 접두사 캐시 영향을 제거하고, 양쪽 환경의 요청 처리 상한을 맞춘다. 최대 문맥 길이는 4,096토큰, 동시에 스케줄링할 수 있는 요청 수는 32개, 한 번에 처리할 토큰 예산은 4,096으로 지정한다. 이 값은 실습의 비교 조건이며 최대 성능을 위한 튜닝 값은 아니다.

접두사 캐시 해제 옵션은 현재 공식 CLI의 `--disable-prefix-caching`을 사용한다. 설치 버전의 `vllm serve --help`에서 `--no-enable-prefix-caching`으로 표시된다면 아래 옵션 문자열을 그 이름으로 바꾼다.

In [ ]:
server_command = [
    "env", f"CUDA_VISIBLE_DEVICES={SERVER_GPUS}",
    *VLLM, "serve", MODEL,
    "--host", "127.0.0.1",
    "--port", "8000",
    "--tensor-parallel-size", str(TENSOR_PARALLEL_SIZE),
    "--dtype", "bfloat16",
    "--max-model-len", "4096",
    "--max-num-seqs", "32",
    "--max-num-batched-tokens", "4096",
    "--disable-prefix-caching",
    "--generation-config", "vllm",
    "--language-model-only",
]
print(shlex.join(server_command))

### 서버 응답 확인

다음 셀은 실행 중인 서버의 모델 이름과 한국어 질문에 대한 응답 본문을 표시한다. 표시된 모델 이름이 설정과 같은지 확인한다. `enable_thinking=False`로 직접 답변을 받는다. 이 호출은 동작 확인이며 성능 측정에 포함하지 않는다.

In [ ]:
with urlopen(f"{BASE_URL}/v1/models") as response:
    models = json.load(response)
print([item["id"] for item in models["data"]])

request = Request(
    f"{BASE_URL}/v1/chat/completions",
    data=json.dumps({
        "model": MODEL,
        "messages": [{
            "role": "user",
            "content": "추론 엔진과 HTTP 서버의 역할을 한국어로 두 문장으로 설명하세요.",
        }],
        "temperature": 0,
        "chat_template_kwargs": {"enable_thinking": False},
        "max_tokens": 128,
    }).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)
with urlopen(request) as response:
    answer = json.load(response)
print(answer["choices"][0]["message"]["content"])

## 4. 측정 지표와 공통 명령

| 지표 | 의미 |
| --- | --- |
| Successful / Failed requests | 성공 요청 수와 실패 요청 수 |
| Output token throughput | 전체 측정 구간의 출력 토큰 수를 시간으로 나눈 값 |
| TTFT | 클라이언트가 요청을 전송한 뒤 첫 출력 토큰을 받기까지의 시간 |
| TPOT | 요청별로 첫 토큰 이후 생성에 걸린 시간을 나머지 출력 토큰 수로 나눈 값 |
| E2EL | 클라이언트가 요청을 전송한 뒤 응답이 완료될 때까지의 시간 |
| p50 / p95 | 측정된 요청 지연시간 분포의 50 / 95백분위수 |

`vllm bench serve`의 스트리밍 측정을 사용한다. 입력은 토큰 길이를 통제한 합성 데이터다. `--ignore-eos`로 조기 종료를 막고, 온도를 0으로 고정한다. 채팅 템플릿을 사용하지 않는 completions 요청이므로 thinking 모드 간 비교가 아니다. 품질 측정과 실제 서비스 트래픽 재현은 이 실험에 포함되지 않는다.

요청 발생률은 제한하지 않고, 동시 요청 상한만 제한한다. 하나가 완료되면 다음 요청을 보낼 수 있는 부하 방식이다. 초당 일정한 요청이 도착하는 실험과는 다르며, 클라이언트에서 전송 순서를 기다린 시간은 기본 TTFT/E2EL에 포함되지 않을 수 있다.

각 조건의 워밍업과 사전 동작 확인은 도구가 본 측정에서 제외한다. 성공 요청 수가 256이고 실패가 0인지, 실제 총 입력·출력 토큰 수가 설정과 맞는지 출력에서 확인한다. 총 출력 토큰 수는 `NUM_PROMPTS × 출력 길이`여야 한다. 합성 프롬프트의 문자열 변환과 재토큰화 때문에 입력 토큰 수는 목표 길이와 조금 다를 수 있다.

`--num-warmups` 등 사용 옵션이 설치된 `vllm bench serve --help`에 있어야 한다. 지원하지 않는 버전은 같은 명령으로 실행할 수 없다. 설치된 GPU 런타임을 노트북에서 자동 변경하지 않는다.

In [ ]:
def benchmark(input_tokens, output_tokens, concurrency):
    command = [
        *VLLM, "bench", "serve",
        "--backend", "openai",
        "--base-url", BASE_URL,
        "--model", MODEL,
        "--dataset-name", "random",
        "--random-input-len", str(input_tokens),
        "--random-output-len", str(output_tokens),
        "--random-range-ratio", "0",
        "--num-prompts", str(NUM_PROMPTS),
        "--num-warmups", "8",
        "--max-concurrency", str(concurrency),
        "--request-rate", "inf",
        "--temperature", "0",
        "--ignore-eos",
        "--percentile-metrics", "ttft,tpot,e2el",
        "--metric-percentiles", "50,95",
    ]
    for _ in range(REPEATS):
        result = subprocess.run(
            command, stdout=subprocess.PIPE, text=True, check=True,
        )
        print(
            f"input_tokens={input_tokens}, output_tokens={output_tokens}, "
            f"concurrency={concurrency}"
        )
        print(result.stdout.split("Serving Benchmark Result", 1)[1])

## 5. 동시 요청 수 변경

입력 512토큰, 출력 128토큰을 고정하고 동시 요청 상한만 1·4·16·32로 바꾼다. 각 조건을 3회 측정한다.

출력 토큰 처리량과 p95 TTFT를 함께 읽는다. 동시 요청이 많아질 때 처리량이 계속 증가하는지, 증가가 줄어드는 구간에서 지연시간은 어떻게 변하는지 확인한다. 배치 처리와 자원 사용의 변화를 관찰하는 실험이며, GPU 사용률 자체를 측정하지는 않는다.

In [ ]:
for concurrency in [1, 4, 16, 32]:
    benchmark(input_tokens=512, output_tokens=128, concurrency=concurrency)

## 6. 입력 길이 변경

동시 요청 상한 4와 출력 128토큰을 고정한다. 입력 길이만 128·512·2,048토큰으로 바꾼다. 각 조건을 3회 측정한다.

입력 길이에 따라 p50·p95 TTFT가 어떻게 변하는지 확인한다. TTFT에는 입력 처리뿐 아니라 서버 대기와 통신도 포함되므로 순수한 GPU prefill 시간과 같다고 해석하지 않는다.

In [ ]:
for input_tokens in [128, 512, 2048]:
    benchmark(input_tokens=input_tokens, output_tokens=128, concurrency=4)

## 7. 출력 길이 변경

동시 요청 상한 4와 입력 512토큰을 고정한다. 출력 길이만 32·128·512토큰으로 바꾼다. 각 조건을 3회 측정한다.

전체 응답시간과 TPOT를 비교한다. 긴 출력을 생성할 때 응답시간과 토큰당 시간이 같은 비율로 변하는지 확인한다. 출력 길이가 늘어나는 조건에서도 처리량만으로 응답성을 판단하지 않는다.

In [ ]:
for output_tokens in [32, 128, 512]:
    benchmark(input_tokens=512, output_tokens=output_tokens, concurrency=4)

## 8. 두 환경의 결과 비교

동일한 모델·정밀도·입출력 길이·동시 요청 조건의 결과끼리 비교한다. 패키지 버전, 드라이버, CPU, GPU 연결 방식이 다르면 그 차이도 결과에 포함된다.

| 비교 | 해석 범위 |
| --- | --- |
| 직접 실행: H100 1개와 B300 1개 | 각 단일 GPU 시스템에서의 같은 작업 처리 성능 |
| 서버: H100 4개(TP 4)와 B300 1개(TP 1) | GPU 수와 통신 비용까지 포함한 서버 구성 간 성능 |
| 같은 환경에서 동시 요청 수 변경 | 해당 요청 길이에서 처리량과 응답 지연의 변화 |
| 같은 환경에서 입력 또는 출력 길이 변경 | 해당 동시 요청 수에서 길이 변화의 영향 |

H100 4개 결과를 4로 나누어 H100 1개 성능으로 해석하지 않는다. TP 4는 GPU별 가중치 메모리와 연산을 나누지만 GPU 간 통신 비용도 발생한다. TP 4가 항상 빠르다고 가정하지 않는다. 이 설정으로 H100 또는 B300의 최대 성능을 측정했다고 주장하지 않는다.

다음 내용을 측정값으로 설명한다.

- 동일한 동시 요청 수에서 처리량과 p95 TTFT가 각각 어떻게 다른가?
- 동시 요청 수 증가에 따른 처리량 증가가 줄어드는 지점은 어디인가?
- 긴 입력과 긴 출력이 TTFT·TPOT·전체 응답시간에 미치는 영향은 어떻게 다른가?
- 세 번의 측정에서 차이가 일관되게 나타나는가?

반복 측정의 p95를 합쳐 전체 요청의 p95로 해석하지 않는다. 각 측정값과 그 변동 범위를 비교한다. 실제 성능 수치는 실행 후 판단한다.

## 9. 종료와 결과 읽기

서버를 실행한 터미널에서 `Ctrl+C`로 서버를 종료한다. 노트북을 닫는 것만으로 별도 터미널의 서버가 종료되지는 않는다.

셀 출력을 보관하려면 노트북을 저장한다. 이후 결과만 읽을 때는 실행 셀을 다시 실행하지 않는다. 새 측정은 회사 GPU를 다시 사용하며, 모델이 없으면 다운로드도 발생한다. 외부 유료 추론 API는 호출하지 않는다.

## 참고 자료

- [Inference Korea 5주차](https://songys.github.io/inference-korea/)
- [Inference Engineering 4장 한국어 참고 자료](https://corazzon.github.io/inference-engineering/ko/04-소프트웨어.html)
- [vLLM 서버 실행](https://docs.vllm.ai/en/stable/cli/serve/)
- [vLLM 서버 벤치마크](https://docs.vllm.ai/en/stable/cli/bench/serve/)
- [vLLM 병렬화](https://docs.vllm.ai/en/stable/serving/parallelism_scaling/)
- [Qwen3.8-27B 모델 카드](https://huggingface.co/Qwen/Qwen3.8-27B)
- [Qwen3.8-27B 모델 설정](https://huggingface.co/Qwen/Qwen3.8-27B/blob/main/config.json)
- [Qwen3.8-27B vLLM 실행 안내](https://recipes.vllm.ai/Qwen/Qwen3.8-27B)
- [PagedAttention 논문](https://arxiv.org/abs/2309.06180)

설명과 실습 코드는 별도로 작성했다. 모델 라이선스는 Apache-2.0이다. 이 노트북에는 교재 본문이나 그림을 복제하지 않았다.